# Day 7 — Single-cell RNA-seq with Scanpy

## PBMC3k — your first complete scRNA-seq analysis

Today we go from raw counts to annotated cell types in one notebook.

**The pipeline** (7 steps, ~15 lines of actual code):
1. Load + inspect the AnnData object
2. Quality control (QC) on cells
3. Filter low-quality cells and lowly-expressed genes
4. Normalize + log-transform
5. Find highly variable genes (HVGs)
6. PCA → neighbors → UMAP → Leiden clustering
7. Find marker genes per cluster → annotate cell types

**Dataset:** 2,700 PBMCs from a healthy donor (10x Genomics public dataset). PBMC = Peripheral Blood Mononuclear Cells = T cells, B cells, NK cells, monocytes, dendritic cells.

> ⚠️ **Colab note:** the install in the next cell may trigger a *Restart session* dialog. Click **Restart**, then re-run from the top.

## 1. Setup

Install Scanpy and import everything we need.

In [ ]:
!pip install scanpy leidenalg --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc

sc.settings.set_figure_params(dpi=80, facecolor='white')
sc.settings.verbosity = 1
print('Scanpy:', sc.__version__)
sc.logging.print_header()

## 2. Load the PBMC3k dataset

Scanpy ships with a built-in copy of this 10x dataset. One line.

In [ ]:
adata = sc.datasets.pbmc3k()
adata

**What is `adata`?** An **AnnData** object — the standard scRNA data container.

| Attribute | What it holds |
|---|---|
| `adata.X` | The count matrix (cells × genes) |
| `adata.obs` | One row per CELL — metadata about each cell |
| `adata.var` | One row per GENE — metadata about each gene |
| `adata.obsm` | Dimensional embeddings (PCA, UMAP) — added later |
| `adata.uns` | Unstructured stuff (clustering results, colors, etc.) |

Throughout this notebook, we keep adding info to the SAME adata object.

In [ ]:
print('Cells x Genes:', adata.shape)
print()
print('First 5 genes:', adata.var_names[:5].tolist())
print('First 5 cells:', adata.obs_names[:5].tolist())

## 3. Quality Control

Before we trust the data, we check three QC metrics for each cell:

- **`n_genes_by_counts`** — how many genes were detected in this cell? (Healthy: 500-5000)
- **`total_counts`** — total UMI count for this cell (sequencing depth per cell)
- **`pct_counts_mt`** — % of reads from mitochondrial genes (high = dying cell)

Scanpy computes these for us with one function.

In [ ]:
# Mark mitochondrial genes
adata.var['mt'] = adata.var_names.str.startswith('MT-')   # human MT genes start with MT-
print('Mitochondrial genes found:', adata.var['mt'].sum())

In [ ]:
# Compute QC metrics
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
adata.obs.head()

Now let's visualize the QC distributions across all 2,700 cells:

In [ ]:
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
             jitter=0.4, multi_panel=True)

In [ ]:
# Joint distribution — look for the 'good cell' cloud and outliers
sc.pl.scatter(adata, x='total_counts', y='pct_counts_mt')
sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts')

**🧐 Questions to discuss:**
1. What's the typical range for `n_genes_by_counts`?
2. Do any cells have suspiciously high `pct_counts_mt` (>10%)? Those are dying cells.
3. Do you see any clear outliers in the scatter plots?

## 4. Filter low-quality cells and lowly-expressed genes

Standard cutoffs for PBMC3k:
- Keep cells with **200–2,500 genes** (not too few, not too many — too many = doublets)
- Keep cells with **< 5% mitochondrial reads**
- Keep genes detected in **at least 3 cells**

In [ ]:
# Filter genes first (Scanpy's convention)
sc.pp.filter_genes(adata, min_cells=3)

# Filter cells by number of genes
sc.pp.filter_cells(adata, min_genes=200)

# Filter cells with extreme values
adata = adata[adata.obs.n_genes_by_counts < 2500, :]
adata = adata[adata.obs.pct_counts_mt < 5, :]

print('After filtering:', adata.shape)

## 5. Normalize + log-transform

**Why normalize?** Cells captured different total amounts of RNA. We normalize each cell to the same total count, then log-transform to compress the wide expression range.

**Tip:** save the raw counts first — we'll need them later for `rank_genes_groups`.

In [ ]:
# Save raw counts (needed for marker gene identification later)
adata.layers['counts'] = adata.X.copy()

# Normalize each cell to 10,000 total counts
sc.pp.normalize_total(adata, target_sum=1e4)

# Log(x+1) transformation
sc.pp.log1p(adata)

# Snapshot of log-normalized values for the raw slot
adata.raw = adata

## 6. Find highly variable genes (HVGs)

Out of ~20,000 genes, most are noise. We pick the most **variable** genes — these carry the cell-type signal. ~2,000 is the typical target.

In [ ]:
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
print('Highly variable genes:', adata.var.highly_variable.sum())
sc.pl.highly_variable_genes(adata)

Keep only the HVGs for downstream analysis:

In [ ]:
adata = adata[:, adata.var.highly_variable]
print('After keeping only HVGs:', adata.shape)

## 7. Scale the data (one more cleanup step)

Center each gene at mean 0, scale to unit variance. Required before PCA. Clip extreme values.

In [ ]:
sc.pp.scale(adata, max_value=10)

## 8. PCA → Neighbors → UMAP

Three lines reduce 2,000 dimensions to 2D — the most beautiful plot in modern biology.

In [ ]:
# Step 1: PCA — reduce to 50 components
sc.tl.pca(adata, svd_solver='arpack')
sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True)

In [ ]:
# Step 2: Build the neighbor graph (each cell connected to its 10 nearest)
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)

# Step 3: UMAP
sc.tl.umap(adata)

**Look at the first UMAP!** Color by total counts to see structure:

In [ ]:
sc.pl.umap(adata, color='total_counts')

## 9. Leiden clustering — find cell groups

On the neighbor graph, Leiden finds dense communities of similar cells. These (hopefully) correspond to cell types.

In [ ]:
sc.tl.leiden(adata, resolution=0.5, flavor='igraph', n_iterations=2, directed=False)

# Plot
sc.pl.umap(adata, color='leiden', legend_loc='on data',
           title='Leiden clusters (resolution=0.5)', frameon=False)

**🧪 Try it yourself:** Re-run with different resolutions and see how the clusters change.

```python
sc.tl.leiden(adata, resolution=0.3, key_added='leiden_0.3', flavor='igraph', n_iterations=2, directed=False)
sc.tl.leiden(adata, resolution=1.0, key_added='leiden_1.0', flavor='igraph', n_iterations=2, directed=False)
sc.pl.umap(adata, color=['leiden_0.3', 'leiden', 'leiden_1.0'], ncols=3)
```

In [ ]:
# Your code here — try different resolutions


## 10. Find marker genes per cluster

For each cluster, find genes that are HIGHER than in all other cells. Those are the markers we'll use to identify cell types.

In [ ]:
sc.tl.rank_genes_groups(adata, 'leiden', method='wilcoxon')
sc.pl.rank_genes_groups(adata, n_genes=10, sharey=False)

Look at the top markers as a table:

In [ ]:
# Show top 10 markers per cluster
pd.DataFrame(adata.uns['rank_genes_groups']['names']).head(10)

## 11. Annotate cell types using known PBMC markers

Compare your top markers to the classic PBMC marker gene reference:

| Cell type | Marker genes |
|---|---|
| **T cells** | CD3D, CD3E, IL7R |
| **B cells** | MS4A1, CD79A, CD19 |
| **CD14 Monocytes** | CD14, LYZ |
| **FCGR3A Monocytes** | FCGR3A, MS4A7 |
| **NK cells** | GNLY, NKG7 |
| **Dendritic** | FCER1A, CST3 |
| **Megakaryocytes** | PPBP |

In [ ]:
# Define a marker gene dictionary
marker_genes = {
    'CD4 T':   ['IL7R', 'CD4'],
    'CD8 T':   ['CD8A', 'CD8B'],
    'B cell':  ['MS4A1', 'CD79A'],
    'Mono CD14':['CD14', 'LYZ'],
    'Mono CD16':['FCGR3A', 'MS4A7'],
    'NK':      ['GNLY', 'NKG7'],
    'DC':      ['FCER1A', 'CST3'],
    'Megakaryocyte': ['PPBP']
}

# Dot plot of markers vs clusters
sc.pl.dotplot(adata, marker_genes, groupby='leiden', dendrogram=True)

**🧐 From the dot plot above, you should be able to match each cluster to a cell type.**

For example, the cluster where IL7R is strong = T cells. The cluster where MS4A1 is strong = B cells.

In [ ]:
# Map clusters to cell types (you'll need to adjust based on YOUR dotplot!)
# Example annotation — DOUBLE-CHECK against your dot plot before using:
cluster_to_celltype = {
    '0': 'CD4 T',
    '1': 'CD14 Mono',
    '2': 'B cell',
    '3': 'CD8 T',
    '4': 'NK',
    '5': 'FCGR3A Mono',
    '6': 'Dendritic',
    '7': 'Megakaryocyte',
}

adata.obs['cell_type'] = adata.obs['leiden'].map(cluster_to_celltype).astype('category')

In [ ]:
# Final UMAP — colored by your annotated cell types!
sc.pl.umap(adata, color='cell_type', legend_loc='on data',
           title='PBMC3k — annotated cell types',
           frameon=False, palette='Set1')

## 12. Final visualizations — the publication-quality plots

These are the four plots every scRNA paper has. Practice each one.

In [ ]:
# UMAP colored by individual marker gene expression
sc.pl.umap(adata, color=['CD3D', 'MS4A1', 'CD14', 'GNLY', 'PPBP', 'FCER1A'],
           ncols=3, frameon=False, cmap='viridis')

In [ ]:
# Violin plot — show CD3D and MS4A1 across cell types
sc.pl.violin(adata, ['CD3D', 'MS4A1'], groupby='cell_type', rotation=45)

In [ ]:
# Heatmap of top markers per cell type
sc.pl.rank_genes_groups_heatmap(adata, n_genes=5, groupby='leiden',
                                  show_gene_labels=True, cmap='viridis')

---
## 🎉 You did it

You just ran a **complete scRNA-seq analysis** — the same workflow used in published Nature papers.

| Step | What you got |
|---|---|
| QC + filtering | Kept ~2,000 healthy cells |
| Normalize + HVG | 2,000 most informative genes |
| PCA + UMAP | 2D embedding of all cells |
| Leiden clustering | ~8 cell-type clusters |
| Marker genes | Identified known PBMC markers |
| Cell type annotation | Labeled each cluster |

## Take-home assignment

Your **final assessment** is a separate notebook: `Day7_Final_Assessment_PBMC68k.ipynb`

Apply everything you learned today to the **PBMC68k_reduced dataset** (700 cells, pre-processed).

5 graded tasks, 100 points total. **Submit by next Friday.**

---

**Thank you for an amazing 4 weekends!** This is the same toolkit professional bioinformaticians use every day. You're ready.